## Setup & Import
### Install and import the necessary libraries for Spark and MongoDb

In [1]:
# Install necessary packages
%pip install pyspark pymongo dnspython

import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, when, desc, rank
from pyspark.sql.window import Window
import pymongo
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

   ---------------------------------------- 0.0/855.8 kB ? eta -:--:--
   ------------ --------------------------- 262.1/855.8 kB ? eta -:--:--
   ------------------------ --------------- 524.3/855.8 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 855.8/855.8 kB 1.3 MB/s  0:00:00

   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [p

ModuleNotFoundError: No module named 'pandas'

## Ingest Data into MongoDB
### Load the CSV file into MongoDB to simulate a real data source

In [12]:
# Connect to MongoDB (Local or Atlas)
# Replace 'your_connection_string' with your actual MongoDB URI
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["diabetes_db"]
collection = db["patients"]

# Load CSV and insert into MongoDB
df_csv = pd.read_csv('diabetes_dataset.csv')
data_dict = df_csv.to_dict(orient='records')

# Clear collection and insert new data
collection.delete_many({}) 
collection.insert_many(data_dict)

print(f"Successfully ingested {len(data_dict)} records into MongoDB.")

Successfully ingested 100000 records into MongoDB.


## Spark Data Loading & Cleaning
### Creat a SparkSession and read data from MongoDB.We define a schema for better accuracy

In [4]:
# Initialize Spark Session with MongoDB Connector
spark = SparkSession.builder \
    .appName("DiabetesProject") \
    .config("spark.mongodb.read.connection.uri", "mongodb://127.0.0.1/diabetes_db.patients") \
    .config("spark.mongodb.write.connection.uri", "mongodb://127.0.0.1/diabetes_db.patients") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

# Read data from MongoDB
df = spark.read.format("mongo").load()

# Data Cleaning: Filter out nulls and cast types
df_clean = df.filter(col("age").isNotNull()) \
             .withColumn("age", col("age").cast("int")) \
             .withColumn("bmi", col("bmi").cast("double")) \
             .withColumn("diabetes", col("diabetes").cast("int"))

df_clean.printSchema()
df_clean.show(5)

Py4JJavaError: An error occurred while calling o56.load.
: org.apache.spark.SparkClassNotFoundException: [DATA_SOURCE_NOT_FOUND] Failed to find the data source: mongo. Make sure the provider name is correct and the package is properly registered and compatible with your Spark version. SQLSTATE: 42K02
	at org.apache.spark.sql.errors.QueryExecutionErrors$.dataSourceNotFoundError(QueryExecutionErrors.scala:764)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:686)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSourceV2(DataSource.scala:745)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:58)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:45)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.$anonfun$resolveOperatorsUpWithPruning$3(AnalysisHelper.scala:139)
	at org.apache.sp

## Spark SQL & Advanced Analysis
### Use Spark SQL to calculate diabetes rates and rank locations using Window Functions

In [ ]:
# Create a temporary view for SQL
df_clean.createOrReplaceTempView("diabetes_data")

# SQL Query: Calculate average BMI and Diabetes rate per location
sql_query = """
    SELECT location, 
           COUNT(*) as total_patients, 
           AVG(bmi) as avg_bmi, 
           AVG(diabetes) * 100 as diabetes_percentage
    FROM diabetes_data
    GROUP BY location
    HAVING total_patients > 100
    ORDER BY diabetes_percentage DESC
"""
analysis_results = spark.sql(sql_query)
analysis_results.show(10)

# Advanced: Using Window Function to rank patients by glucose level within each gender
windowSpec = Window.partitionBy("gender").orderBy(desc("blood_glucose_level"))
df_ranked = df_clean.withColumn("rank", rank().over(windowSpec))

print("Top 3 patients with highest glucose levels per gender:")
df_ranked.filter(col("rank") <= 3).select("gender", "blood_glucose_level", "rank").show()

## Performance & Indexing
### Create an index in MongoDB to optimize queries and compare execution time

In [ ]:
import time

# 1. Query without index (Benchmark)
start_time = time.time()
res1 = list(collection.find({"location": "Alabama"}))
print(f"Time without index: {time.time() - start_time:.4f} seconds")

# 2. Create index
collection.create_index([("location", pymongo.ASCENDING)])
print("Index created on 'location' column.")

# 3. Query with index
start_time = time.time()
res2 = list(collection.find({"location": "Alabama"}))
print(f"Time with index: {time.time() - start_time:.4f} seconds")

## Visualization
### Convert results to Pandas and plot a graph to show insights

In [ ]:
# Convert to Pandas for plotting
pdf = df_clean.select("age", "diabetes", "bmi").toPandas()

plt.figure(figsize=(10, 6))
sns.histplot(data=pdf, x="age", hue="diabetes", multiple="stack", bins=20)
plt.title("Diabetes Distribution by Age")
plt.xlabel("Age")
plt.ylabel("Number of Patients")
plt.show()

# Correlation heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(pdf.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix: Age, BMI, and Diabetes")
plt.show()